<table style="width: 100%; border: none; background: #f8f9fa; padding: 20px; border-radius: 10px;">
    <tr>
        <td style="width: 22%; border: none; vertical-align: middle; text-align: center;">
            <img src="https://www.inf.ucv.cl/wp-content/uploads/2020/05/logo_escuela.jpg" width="170">
        </td>
        <td style="width: 56%; border: none; vertical-align: middle; text-align: center; padding: 0 20px;">
            <h1 style="font-size: 28px; color: #1a3b5c; margin-bottom: 16px; border-bottom: 2px solid #1a3b5c; padding-bottom: 8px;">Tarea 3: Boosting, Stacking, Interpretabilidad y Ética</h1>
            <p style="font-size: 16px; margin: 8px 0;"><strong>Pontificia Universidad Catolica de Valparaiso</strong><br>
            <strong>Escuela de Ingenieria Informatica</strong></p>
            <p style="font-size: 14px; color: #333; margin: 12px 0;"><strong>OII432 Machine Learning</strong><br>
            <strong>Semestre: 2-2026</strong><br>
            <strong>Profesor: Carlos Valle</strong></p>
        </td>
        <td style="width: 22%; border: none; vertical-align: middle; text-align: center;">
            <img src="https://upload.wikimedia.org/wikipedia/commons/c/cb/PUCV_Escudo2016.svg" width="110">
        </td>
    </tr>
</table>

# Tarea 3: Boosting, stacking, interpretabilidad y ética

Objetivo: construir sobre datos reales los ensembles secuenciales del capítulo 3 (AdaBoost, gradient boosting y sus implementaciones modernas) y el stacking, y luego hacerse cargo del modelo como pide el capítulo 4: explicarlo, auditar si trata distinto a distintos grupos, y medir cuánto revela de las personas que lo entrenaron.

Temas: AdaBoost y la pérdida exponencial, el *edge* y el paso de búsqueda lineal, gradient boosting con pseudo-residuos y paso de Newton, tasa de aprendizaje y profundidad, XGBoost, LightGBM y CatBoost, target statistics y su fuga, stacking con predicciones out-of-fold, importancia por permutación, PDP, ICE y ALE, valores de Shapley y SHAP, LIME, contrafactuales, criterios de equidad y el teorema de imposibilidad, post-procesamiento por grupo, k-anonimato y privacidad diferencial.

Formalidades

* Los códigos de referencia son solo orientativos.
* Entregar un Jupyter Notebook (.ipynb) con código, análisis e interpretaciones en celdas markdown.
* Trabajo en grupos de 3 o 4. La copia parcial o total será penalizada con nota 1.
* Entregable subido a un repositorio privado de GitHub. Enviar el link a <carlos.valle@pucv.cl> (Asunto: [OII432-2-2026 Tarea3]) e invitar como colaborador al usuario **cvalle231**.
* Fecha de entrega: **lunes 23 de noviembre**.
* Deben estar preparados para presentar su tarea en las clases del **viernes 20 de noviembre**.

La tarea tiene dos preguntas de igual ponderación (50% cada una). La nota final es el promedio de ambas.

Librerías adicionales a las de la Tarea 2: `xgboost`, `lightgbm`, `catboost` y `shap` (`pip install xgboost lightgbm catboost shap`).

# Pregunta 1: (50%) Corregir los errores de a uno: boosting y stacking

<p align="center"><img src="https://www2.census.gov/library/photos/decennial/1980/1980-census-enumeration.jpg" width="600"></p>

En la Tarea 2 promediamos árboles profundos, entrenados en paralelo e independientes, para bajarles la varianza. Aquí haremos lo contrario: modelos **débiles**, entrenados **en secuencia**, cada uno concentrado en lo que el conjunto todavía hace mal.

Trabajaremos con **Adult**, el extracto del censo de Estados Unidos de 1994: 48.842 personas, 14 atributos y una etiqueta que indica si la persona ganaba más de US$50.000 al año. Tres propiedades lo hacen un buen laboratorio para este capítulo.

Primero, mezcla atributos **numéricos y categóricos**, algunos con muchas categorías (`native-country` tiene 41): el terreno donde las tres librerías modernas toman caminos distintos.

Segundo, tiene **datos faltantes** en tres columnas categóricas, que XGBoost y LightGBM saben enviar por una rama aprendida sin imputar nada.

Tercero, un modelo lineal ya funciona razonablemente bien, así que lo que el boosting agregue habrá que ganárselo, y medirlo.

El dataset trae atributos sensibles (sexo, raza). En esta pregunta no los analizaremos: de eso se trata la Pregunta 2.

Esta parte se evalúa sobre 58 puntos, repartidos entre las subpreguntas según el trabajo que exige cada una. La nota se obtiene como:

$$\text{Nota Pregunta 1} = \frac{\text{puntos obtenidos}}{58} \times 6 + 1.$$

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import expit, logit

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              HistGradientBoostingClassifier)
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_palette("deep")
np.random.seed(42)

# Parte 1: Los datos y la vara que hay que superar

## 1. Carga, partición y líneas base

### 1a) (2 pts) Carga Adult y descríbelo

La celda de abajo descarga el dataset desde OpenML y, si esa descarga falla, desde una copia en GitHub. Además normaliza las dos versiones para que queden iguales: las categóricas como texto sin espacios, y los faltantes (que en el archivo original vienen como `'?'`) como `NaN`.

Deja los atributos en `X` y la etiqueta en `y`, codificada como 1 si la persona gana más de US$50.000 y 0 si no. Reporta cuántas personas hay, la proporción de positivos, el número de categorías de cada atributo categórico y la fracción de faltantes de cada columna que los tenga.

In [ ]:
COLS = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
        "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
        "hours-per-week", "native-country", "class"]
NUM = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
CAT = [c for c in COLS[:-1] if c not in NUM]
URL_RESPALDO = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/adult-all.csv"

try:
    df = fetch_openml("adult", version=2, as_frame=True, parser="auto").frame
except Exception:
    df = pd.read_csv(URL_RESPALDO, header=None, names=COLS)

def limpiar(v):
    if not isinstance(v, str):
        return np.nan
    v = v.strip().rstrip(".")
    return np.nan if v == "?" else v

for c in CAT + ["class"]:
    df[c] = df[c].astype(object).map(limpiar)
df[NUM] = df[NUM].astype(float)

# TODO: X = ...   (todas las columnas salvo "class")
# TODO: y = ...   (np.array de enteros: 1 si class == ">50K")

print(f"Personas: {len(X)} | Positivos: {100 * y.mean():.2f}%")
# TODO: numero de categorias por atributo categorico y fraccion de faltantes por columna

### 1b) (2 pts) Particiona y prepara tres representaciones

Usa una partición 60/20/20 estratificada. Cada familia de modelos de esta pregunta necesita los datos de forma distinta:

* **one-hot denso** (`O_*`): para sklearn y para sus propias implementaciones de AdaBoost y gradient boosting. Los faltantes quedan como una categoría más (`dummy_na=True`).
* **dtype `category`** (`C_*`): para XGBoost y LightGBM, que reciben las categóricas en forma nativa y dejan los faltantes como `NaN`.
* **texto con `"NA"` explícito** (`S_*`): para CatBoost, que no acepta `NaN` en columnas categóricas.

Completa las tres representaciones y reporta cuántas columnas tiene cada una.

In [ ]:
X_tr, X_temp, y_tr, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_te, y_val, y_te = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# (i) one-hot denso, con las columnas definidas por entrenamiento
O_tr = pd.get_dummies(X_tr, columns=CAT, dummy_na=True, dtype=float)
# TODO: O_val, O_te con exactamente las columnas de O_tr (reindex(columns=O_tr.columns, fill_value=0))
# TODO: pasar las tres a np.array

# (ii) category, con las categorias vistas en entrenamiento
def a_category(d, ref):
    out = d.copy()
    for c in CAT:
        out[c] = pd.Categorical(out[c], categories=sorted(ref[c].dropna().unique()))
    return out
C_tr, C_val, C_te = (a_category(d, X_tr) for d in (X_tr, X_val, X_te))

# (iii) texto con "NA" explicito
# TODO: S_tr, S_val, S_te = ...

# TODO: reportar el numero de columnas de cada representacion

### 1c) (2 pts) Las líneas base

Ajusta una regresión logística regularizada (sobre el one-hot estandarizado) y un Random Forest de 300 árboles. Reporta el AUC y el log-loss de ambos en validación. Esa tabla es la vara: todo lo que sigue tiene que superarla.

In [ ]:
# TODO: logistica regularizada dentro de un pipeline con StandardScaler
# TODO: Random Forest de 300 arboles
# TODO: tabla con AUC y log-loss en validacion

# Parte 2: AdaBoost, a mano

## 2. Repesar los casos

### 2a) (4 pts) Implementa AdaBoost con stumps

Usa etiquetas $y_m \in \{-1, +1\}$ y, como aprendiz débil, `DecisionTreeClassifier(max_depth=1)` ajustado con `sample_weight`. En cada ronda calcula el error ponderado $\varepsilon_n$, el voto $\alpha_n = \tfrac12 \log\frac{1-\varepsilon_n}{\varepsilon_n}$ y la actualización de los pesos, tal como en la diapositiva. Entrena 200 rondas sobre `O_tr`, reporta el error y el AUC en validación (usando $F(\boldsymbol{x}) = \sum_n \alpha_n f_n(\boldsymbol{x})$ como puntaje), y compara tus $\alpha_n$ con los `estimator_weights_` de `AdaBoostClassifier` con el mismo stump y `learning_rate=1`: deberías encontrar una razón constante.

In [ ]:
s_tr = np.where(y_tr == 1, 1, -1)
s_val = np.where(y_val == 1, 1, -1)

def adaboost(X, s, N=200):
    M = len(s)
    w = np.full(M, 1 / M)
    stumps, alphas, epsilons = [], [], []
    for n in range(N):
        stump = DecisionTreeClassifier(max_depth=1).fit(X, s, sample_weight=w)
        pred = stump.predict(X)
        # TODO: error ponderado eps
        # TODO: voto alpha
        # TODO: w <- w * exp(-alpha * s * pred), y normalizar
        # stumps.append(stump); alphas.append(alpha); epsilons.append(eps)
        pass
    return stumps, np.array(alphas), np.array(epsilons), w

def F_adaboost(stumps, alphas, X):
    # TODO: suma ponderada de las predicciones de los stumps
    pass

stumps, alphas, epsilons, w_final = adaboost(O_tr, s_tr)

# TODO: error y AUC en validacion
# TODO: AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=200, learning_rate=1.0)
# TODO: razon entre estimator_weights_ y tus alphas

### 2b) (2 pts) Verifica la propiedad de la mitad

La diapositiva afirma que, recién actualizados los pesos, los casos mal clasificados por el stump de esa ronda suman **exactamente la mitad** del peso total. Modifica tu implementación para registrar ese peso en cada ronda y reporta la máxima desviación respecto de $0{,}5$ en las 200 rondas.

In [ ]:
# TODO: registrar, despues de normalizar, el peso total de los casos que el stump de la ronda clasifico mal
# TODO: maxima desviacion respecto de 0.5

### 2c) (2 pts) La trayectoria del entrenamiento

Grafica, contra el número de ronda, el error ponderado $\varepsilon_n$ del stump de cada ronda y, en otro panel, el error del conjunto acumulado $\operatorname{sign}(F_n)$ en entrenamiento y en validación.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
# TODO: eps_n contra n, con la linea horizontal en 0.5
# TODO: error del conjunto acumulado en train y validacion contra n
plt.show()

### 2d) (2 pts) Stumps cada vez peores, conjunto cada vez mejor

En 2c) el error ponderado de cada nuevo stump sube hacia $1/2$ mientras el error del conjunto baja. Explica por qué no hay contradicción, usando la propiedad de 2b) y la lectura de AdaBoost como descenso por coordenadas: ¿qué le pasa al *edge* $r_n = 1 - 2\varepsilon_n$ y al paso $\alpha_n = \tanh^{-1}(r_n)$?

Respuesta 2d): completar aquí

### 2e) (3 pts) Ruido en las etiquetas

Invierte la etiqueta del 10% de los casos de entrenamiento, elegidos al azar con semilla fija, y vuelve a correr tus 200 rondas. Al final, reporta qué fracción del peso total quedó sobre los casos invertidos y qué fracción del 1% de casos con mayor peso son casos invertidos.

In [ ]:
rng = np.random.default_rng(0)
invertidos = rng.random(len(s_tr)) < 0.10
s_ruido = s_tr.copy()
s_ruido[invertidos] *= -1

# TODO: correr adaboost sobre (O_tr, s_ruido) y tomar los pesos finales
# TODO: fraccion del peso total sobre los casos invertidos
# TODO: fraccion de invertidos dentro del 1% de casos con mayor peso

### 2f) (2 pts) ¿Qué haría la pérdida logística con esos casos?

En AdaBoost el peso de un caso es proporcional a $e^{-y_m F(\boldsymbol{x}_m)}$. En gradient boosting con log-loss, lo que cada caso aporta al siguiente árbol es su pseudo-residuo $y_m - p_m$, acotado entre $-1$ y $1$. Con lo que viste en 2e) y el gráfico de pérdidas de la diapositiva, explica cuál de los dos es más frágil ante etiquetas mal puestas y por qué.

Respuesta 2f): completar aquí

# Parte 3: Gradient boosting, a mano

## 3. Ajustar los residuos

### 3a) (4 pts) Implementa gradient boosting para log-loss

Con $y \in \{0, 1\}$ y $F$ en log-odds: parte de $F_0 = \operatorname{logit}(\bar y)$; en cada ronda calcula el pseudo-residuo $r_m = y_m - p_m$ con $p_m = \sigma(F_{n-1}(\boldsymbol{x}_m))$, ajusta un `DecisionTreeRegressor(max_depth=3)` a $(\boldsymbol{x}_m, r_m)$ y reemplaza el valor de cada hoja por el **paso de Newton** de la diapositiva, $\gamma_j = \sum_{R_j}(y_m - p_m) \big/ \sum_{R_j} p_m(1-p_m)$. Usa `tree.apply` para saber en qué hoja cae cada caso. Entrena 300 rondas con $\nu = 0{,}1$ y registra el log-loss de entrenamiento y de validación en cada ronda.

In [ ]:
def gb_logloss(Xtr, ytr, Xva, yva, N=300, nu=0.1, newton=True):
    F_tr = np.full(len(ytr), logit(ytr.mean()))
    F_va = np.full(len(yva), logit(ytr.mean()))
    hist_tr, hist_va = [], []
    for n in range(N):
        p = expit(F_tr)
        r = ytr - p
        arbol = DecisionTreeRegressor(max_depth=3).fit(Xtr, r)
        hojas_tr, hojas_va = arbol.apply(Xtr), arbol.apply(Xva)
        if newton:
            # TODO: gamma de Newton por hoja, y actualizar F_tr y F_va con nu * gamma
            pass
        else:
            # TODO: usar el valor que el arbol ya tiene en la hoja (arbol.predict)
            pass
        hist_tr.append(log_loss(ytr, expit(F_tr)))
        hist_va.append(log_loss(yva, expit(F_va)))
    return hist_tr, hist_va

hist_tr_n, hist_va_n = gb_logloss(O_tr, y_tr, O_val, y_val, newton=True)

### 3b) (2 pts) Newton contra el promedio del residuo

Repite el entrenamiento usando como valor de hoja el promedio del pseudo-residuo (lo que el árbol de regresión ya trae), y grafica las dos curvas de log-loss de validación. Comenta cuál baja más rápido y qué información usa el paso de Newton que el promedio ignora.

In [ ]:
# TODO: gb_logloss(..., newton=False)
# TODO: graficar las dos curvas de validacion

### 3c) (3 pts) Tasa de aprendizaje contra número de árboles

Usa `HistGradientBoostingClassifier(max_depth=3, early_stopping=False)` con $\nu \in \{1;\ 0{,}3;\ 0{,}1;\ 0{,}03\}$ y hasta 2.000 iteraciones. Con `staged_predict_proba`, grafica el log-loss de validación contra el número de árboles para las cuatro tasas, y reporta para cada una el número de árboles óptimo y el log-loss mínimo.

A diferencia del bosque de la Tarea 2, aquí agregar árboles sí puede empeorar el modelo: el gráfico debería mostrarlo.

In [ ]:
tasas = [1, 0.3, 0.1, 0.03]

plt.figure(figsize=(11, 4.5))
for nu in tasas:
    # TODO: ajustar, recorrer staged_predict_proba sobre O_val y guardar el log-loss
    pass
plt.xscale("log"); plt.xlabel("numero de arboles"); plt.ylabel("log-loss (validacion)")
plt.legend(); plt.show()

# TODO: tabla con N optimo y log-loss minimo por tasa

### 3d) (2 pts) La profundidad es el orden de interacción

Con $\nu = 0{,}1$, compara profundidades $1, 2, 4$ y $6$, cada una con su número óptimo de árboles en validación. Reporta el log-loss mínimo de cada una y comenta qué clase de modelo es el boosting con stumps (profundidad 1) y qué interacciones no puede representar.

In [ ]:
# TODO: para cada profundidad, log-loss minimo en validacion y N optimo

# Parte 4: Las tres librerías modernas

## 4. XGBoost, LightGBM y CatBoost

### 4a) (3 pts) El primer corte de XGBoost, a mano

Entrena un `XGBClassifier` de un solo árbol, con `max_depth=1`, `learning_rate=1`, `reg_lambda=1`, `min_child_weight=0`, `tree_method="exact"` y `base_score` igual a la proporción de positivos de entrenamiento. Lee el corte con `get_booster().trees_to_dataframe()`. Luego, con $g_m = p_0 - y_m$ y $h_m = p_0(1-p_0)$, calcula a mano $G_L, H_L, G_R, H_R$, los valores de hoja $-G/(H+\lambda)$ y la ganancia del corte, y verifica que coinciden con lo que reporta XGBoost en las columnas `Gain`, `Cover` y en las hojas. Ojo: compara la ganancia con y sin el factor $\tfrac12$ de la diapositiva, y di qué convención usa la librería.

In [ ]:
p0 = y_tr.mean()
primer = xgb.XGBClassifier(n_estimators=1, max_depth=1, learning_rate=1, reg_lambda=1,
                           min_child_weight=0, tree_method="exact", base_score=p0)
primer.fit(O_tr, y_tr)
tabla = primer.get_booster().trees_to_dataframe()
print(tabla[["Node", "Feature", "Split", "Gain", "Cover"]])

# TODO: columna y umbral del corte (Feature es 'f<indice>' si O_tr es un array, o el nombre de la columna si es DataFrame)
# TODO: g, h, y las sumas por lado
# TODO: valores de hoja y ganancia a mano, comparados con la tabla

### 4b) (3 pts) Las tres librerías sobre los mismos datos

Entrena XGBoost (`tree_method="hist"`, `enable_categorical=True`, sobre `C_*`), LightGBM (sobre `C_*`) y CatBoost (con `cat_features=CAT`, sobre `S_*`), todos con tasa $0{,}05$, hasta 3.000 iteraciones y parada temprana de 100 rondas sobre validación. Reporta en una tabla el AUC, el log-loss, el número de iteraciones elegido y el tiempo de entrenamiento de cada uno.

In [ ]:
resultados = []

# TODO: XGBoost con early_stopping_rounds=100 y eval_set=[(C_val, y_val)]
# TODO: LightGBM con callbacks=[lgb.early_stopping(100)]
# TODO: CatBoost con early_stopping_rounds=100 y eval_set=(S_val, y_val)

# print(pd.DataFrame(resultados))

### 4c) (2 pts) Histogramas: ¿cuánta resolución hace falta?

Repite el LightGBM de 4b) con `max_bin` $\in \{3, 7, 15, 63, 255\}$. Reporta el log-loss de validación, las iteraciones elegidas y el tiempo, y comenta a partir de cuántos bins deja de importar la resolución en estos datos.

In [ ]:
# TODO: barrido de max_bin con parada temprana

### 4d) (3 pts) La fuga de las target statistics

Construye una categórica de alta cardinalidad cruzando cinco atributos: `occupation`, `native-country`, `education`, `marital-status` y `workclass` (con los faltantes como `"NA"`). Reporta cuántas categorías tiene y qué fracción de los casos de entrenamiento cae en una categoría que aparece una sola vez.

Codifícala con el promedio suavizado de la diapositiva ($a = 1$, prior $p = \bar y$ de entrenamiento) de dos maneras:

* **codicioso**: cada caso de entrenamiento usa la estadística calculada con todo el entrenamiento, incluido él mismo;
* **out-of-fold**: con $K = 5$ folds, cada caso usa la estadística calculada en los otros cuatro.

En validación ambas usan la estadística de todo el entrenamiento, y una categoría nueva recibe el prior. Entrena un LightGBM (300 árboles, tasa $0{,}05$) con los atributos numéricos más la codificación, y reporta el AUC de entrenamiento y de validación para cada versión.

In [ ]:
cruces = ["occupation", "native-country", "education", "marital-status", "workclass"]

def clave(d):
    return d[cruces].fillna("NA").astype(str).apply(lambda r: "|".join(r), axis=1)

k_tr, k_val = clave(X_tr), clave(X_val)
# TODO: numero de categorias y fraccion de casos de train en categorias unicas

a, prior = 1, y_tr.mean()
# TODO: codificacion codiciosa para train
# TODO: codificacion out-of-fold (StratifiedKFold de 5) para train
# TODO: codificacion de validacion con todo el entrenamiento (categorias nuevas -> prior)

# TODO: LightGBM con NUM + codificacion, AUC de train y validacion para cada version

### 4e) (2 pts) Lo que CatBoost resolvió

Explica la brecha que viste en 4d) con el caso extremo de la diapositiva. Luego explica por qué las *ordered target statistics* de CatBoost evitan el problema igual que tu versión out-of-fold, y por qué el mismo defecto reaparece en los residuos de cualquier boosting (el *prediction shift* que corrige el *ordered boosting*).

Respuesta 4e): completar aquí

# Parte 5: Stacking

## 5. Aprender a combinar

### 5a) (3 pts) La matriz out-of-fold

Toma tres modelos base: la logística de 1c), el Random Forest de 1c) y un LightGBM de 300 árboles con tasa $0{,}05$, los tres sobre el one-hot. Con `cross_val_predict` (5 folds estratificados) obtén la matriz $Z$ de probabilidades out-of-fold en entrenamiento. Después reajusta cada modelo base con todo el entrenamiento para obtener sus predicciones en validación. Reporta la matriz de correlación de las columnas de $Z$.

In [ ]:
base = {
    "logistica": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "bosque": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
    "lightgbm": lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
Z_tr, Z_val = {}, {}

for nombre, modelo in base.items():
    # TODO: Z_tr[nombre] = probabilidades out-of-fold
    # TODO: reajustar en todo train y Z_val[nombre] = probabilidades en validacion
    pass

# Z_tr, Z_val = pd.DataFrame(Z_tr), pd.DataFrame(Z_val)
# TODO: matriz de correlacion de Z_tr

### 5b) (2 pts) El meta-learner

Ajusta una regresión logística sobre $\operatorname{logit}(Z)$ (recorta las probabilidades a $[10^{-4}, 1-10^{-4}]$ antes de aplicar el logit). Reporta sus coeficientes y su AUC y log-loss en validación, junto a los de cada modelo base y a los del promedio simple de las tres probabilidades.

In [ ]:
def a_logit(Z, eps=1e-4):
    return logit(np.clip(Z, eps, 1 - eps))

# TODO: meta-learner sobre a_logit(Z_tr), coeficientes y metricas en a_logit(Z_val)
# TODO: tabla comparativa con los tres modelos base y el promedio simple

### 5c) (2 pts) La forma equivocada

Construye ahora la matriz de entrenamiento del meta-learner con las predicciones **dentro de muestra** (cada modelo base ajustado con todo el entrenamiento y evaluado sobre ese mismo entrenamiento). Ajusta el mismo meta-learner y reporta sus coeficientes y su log-loss en validación.

In [ ]:
# TODO: predicciones en muestra de cada modelo base sobre O_tr
# TODO: meta-learner sobre ellas, coeficientes y log-loss en validacion

### 5d) (2 pts) ¿Qué aprendió el meta-learner equivocado?

Explica los coeficientes de 5c) mirando el AUC dentro de muestra de cada modelo base, y relaciona este error con la fuga de 4d): ¿qué tienen en común?

Respuesta 5d): completar aquí

# Parte 6: La decisión

## 6. El modelo final

### 6a) (2 pts) Elige y reporta

Elige un modelo usando **solo** validación y evalúalo una única vez sobre prueba. Reporta AUC, log-loss y accuracy con umbral 0,5, junto a las mismas métricas de las dos líneas base de 1c) sobre prueba.

In [ ]:
# TODO: modelo elegido en validacion, evaluacion unica en prueba y tabla comparativa

### 6b) (2 pts) Bagging contra boosting, en el vocabulario del curso

Con los números de esta pregunta y los de la Tarea 2: ¿qué componente del error reduce principalmente cada familia, y por qué eso explica que el bosque use árboles profundos y el boosting árboles chicos? Responde también por qué en 3c) agregar árboles podía empeorar el modelo, mientras que en el bosque no.

Respuesta 6b): completar aquí

# Pregunta 2: (50%) Auditar un puntaje de riesgo: interpretabilidad y ética

<p align="center"><img src="https://upload.wikimedia.org/wikipedia/commons/e/e7/Legal_Gavel_(27571702173).jpg" width="600"></p>

COMPAS es un puntaje comercial que estima el riesgo de reincidencia de una persona detenida, y que en varios estados de Estados Unidos se usa al decidir fianzas y condenas. En 2016 ProPublica publicó los datos de las personas evaluadas en el condado de Broward, Florida, entre 2013 y 2014, junto con lo que les pasó en los dos años siguientes, y denunció que el puntaje se equivocaba de forma distinta según la raza. La empresa respondió que el puntaje estaba calibrado por grupo. Es el caso canónico del capítulo: las dos partes tenían razón, y el teorema de imposibilidad explica por qué.

Trabajaremos con esos datos. Tres propiedades los hacen el laboratorio exacto de este capítulo.

Primero, traen el **puntaje real** (`decile_score`, de 1 a 10) además del desenlace, así que podemos auditar una herramienta que efectivamente se usó y compararla con un modelo propio.

Segundo, las **prevalencias difieren** entre grupos, que es la condición del teorema de imposibilidad: lo veremos con números, no solo con álgebra.

Tercero, la etiqueta es un **proxy**: `two_year_recid` registra si la persona fue *arrestada* de nuevo en dos años, no si volvió a delinquir. Esa diferencia, que el capítulo llama sesgo de etiqueta, no se corrige con ningún modelo.

Una advertencia antes de empezar: son registros judiciales de personas reales. La celda de carga elimina los nombres. Trabajen solo con resultados agregados y no intenten, en ningún punto de la tarea, identificar a alguien.

Siguiendo el análisis de ProPublica, nos restringiremos a las personas registradas como `African-American` y `Caucasian`: los demás grupos son demasiado chicos para estimar tasas por grupo con un error razonable (recuerden el error estándar del recall en la Tarea 2).

Esta parte se evalúa sobre 55 puntos, repartidos entre las subpreguntas según el trabajo que exige cada una. La nota se obtiene como:

$$\text{Nota Pregunta 2} = \frac{\text{puntos obtenidos}}{55} \times 6 + 1.$$

In [ ]:
import itertools
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

import lightgbm as lgb
import shap

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_palette("deep")
np.random.seed(42)

# Parte 1: Los datos y el modelo a explicar

## 1. Carga, filtros y un modelo propio

### 1a) (2 pts) Carga los datos y describe los dos grupos

La celda de abajo descarga el archivo publicado por ProPublica en GitHub, elimina los nombres y aplica los mismos filtros de su análisis. Restringe a los dos grupos y reporta, para cada uno, cuántas personas hay, la proporción que reincidió según `two_year_recid` y la proporción clasificada por COMPAS como riesgo medio o alto (`decile_score` $\ge 5$).

In [ ]:
URL = "https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv"
raw = pd.read_csv(URL)
raw = raw.drop(columns=["name", "first", "last"])

# Filtros de ProPublica: evaluacion cercana al arresto, desenlace conocido, delito no ordinario
raw = raw[raw["days_b_screening_arrest"].between(-30, 30)
          & (raw["is_recid"] != -1)
          & (raw["c_charge_degree"] != "O")
          & (raw["score_text"] != "N/A")]

GRUPOS = ["African-American", "Caucasian"]
# TODO: df = raw restringido a GRUPOS (usa .copy())
# TODO: por grupo: tamano, tasa de reincidencia y fraccion con decile_score >= 5

### 1b) (2 pts) Un modelo con siete variables, sin la raza

Usa como atributos `age`, `sex`, `priors_count`, `juv_fel_count`, `juv_misd_count`, `juv_other_count` y `c_charge_degree`, con `sex` y `c_charge_degree` codificadas como 0/1. La raza **no** entra al modelo. Particiona 60/20/20 estratificando por la combinación de etiqueta y grupo, entrena el LightGBM indicado y compara su AUC en validación con el AUC del `decile_score` de COMPAS sobre las mismas personas.

In [ ]:
ATRIB = ["age", "sex", "priors_count", "juv_fel_count", "juv_misd_count",
         "juv_other_count", "c_charge_degree"]

X = df[ATRIB].copy()
X["sex"] = (X["sex"] == "Male").astype(int)
X["c_charge_degree"] = (X["c_charge_degree"] == "F").astype(int)
X = X.astype(float)
y = df["two_year_recid"].values
grupo = (df["race"] == "African-American").astype(int).values   # 1 = African-American
compas = df["decile_score"].values

estrato = 2 * y + grupo
idx = np.arange(len(y))
i_tr, i_tmp = train_test_split(idx, test_size=0.4, random_state=42, stratify=estrato)
i_val, i_te = train_test_split(i_tmp, test_size=0.5, random_state=42, stratify=estrato[i_tmp])
X_tr, X_val, X_te = X.iloc[i_tr], X.iloc[i_val], X.iloc[i_te]
y_tr, y_val, y_te = y[i_tr], y[i_val], y[i_te]

modelo = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=8,
                            min_child_samples=30, verbose=-1)
# TODO: ajustar en train
# TODO: AUC del modelo y de compas en validacion

# Parte 2: ¿Qué usa el modelo, en general?

## 2. Permutación, PDP, ICE y ALE

### 2a) (3 pts) Importancia por permutación, a mano

Implementa la importancia por permutación sobre **validación**, con el AUC como métrica y 20 permutaciones por atributo. Grafica la caída media del AUC de cada atributo con barras de error de una desviación estándar.

In [ ]:
def importancia_permutacion(modelo, X, y, R=20, semilla=0):
    rng = np.random.default_rng(semilla)
    base = roc_auc_score(y, modelo.predict_proba(X)[:, 1])
    resultado = {}
    for col in X.columns:
        caidas = []
        for _ in range(R):
            # TODO: copiar X, permutar la columna col y medir la caida del AUC
            pass
        resultado[col] = (np.mean(caidas), np.std(caidas))
    return pd.DataFrame(resultado, index=["media", "desv"]).T

# TODO: calcular y graficar con barras de error

### 2b) (3 pts) PDP e ICE de `priors_count`, a mano

Calcula la dependencia parcial de la probabilidad predicha en función de `priors_count` sobre una grilla de 0 a 20, promediando sobre validación. Superpón las curvas ICE de 100 personas de validación elegidas al azar, **centradas** en su valor en `priors_count = 0`, junto con el PDP centrado del mismo modo.

In [ ]:
grilla = np.arange(0, 21)
muestra = X_val.sample(100, random_state=0)

# TODO: matriz ICE de forma (100, len(grilla)) con la probabilidad predicha
# TODO: PDP = promedio sobre todo X_val en cada valor de la grilla
# TODO: centrar ICE y PDP en el primer punto de la grilla y graficar

### 2c) (3 pts) ALE de `age`, a mano

Implementa el estimador ALE por ventanas de la diapositiva para `age`, con los bordes de ventana en los deciles de `age` en validación y la predicción en log-odds (`modelo.predict(X, raw_score=True)`). Centra la curva y superpónla con el PDP de `age` evaluado en los mismos bordes y centrado igual.

In [ ]:
f_raw = lambda Z: modelo.predict(Z, raw_score=True)
bordes = np.quantile(X_val["age"], np.linspace(0, 1, 11))

# TODO: para cada ventana [z_{k-1}, z_k], promedio de f(z_k, x_-j) - f(z_{k-1}, x_-j)
#       solo sobre los casos de validacion cuya edad cae en la ventana; acumular
# TODO: PDP de age en los mismos bordes
# TODO: centrar ambas curvas y graficar

### 2d) (2 pts) ¿Difieren el PDP y el ALE?

Compara las dos curvas de 2c). Calcula la correlación de `age` con cada uno de los otros atributos en validación y usa esos números para explicar por qué aquí difieren tanto o tan poco, y en qué tipo de dato esperarías que se separaran.

In [ ]:
# TODO: correlacion de age con los demas atributos

Respuesta 2d): completar aquí

# Parte 3: ¿Por qué predijo esto para esta persona?

## 3. Valores de Shapley y SHAP

### 3a) (4 pts) Shapley exacto, por fuerza bruta

Toma la persona de validación con mayor probabilidad predicha. Con un fondo de 200 casos de entrenamiento, define el valor de una coalición como la versión **intervencional** de la diapositiva: la predicción en log-odds promediada sobre el fondo, con los atributos de $S$ fijos en los valores de esa persona y el resto tomados del fondo. Calcula $v(S)$ para las $2^7 = 128$ coaliciones y obtén los siete $\phi_j$ con la fórmula de subconjuntos.

In [ ]:
fondo = X_tr.sample(200, random_state=0)
caso = X_val.iloc[[np.argmax(modelo.predict_proba(X_val)[:, 1])]]
I = len(ATRIB)

def valor(S):
    Z = fondo.copy()
    # TODO: fijar en Z las columnas de S con los valores de caso
    # TODO: devolver el promedio de f_raw(Z)
    pass

v = {S: valor(S) for r in range(I + 1) for S in itertools.combinations(range(I), r)}

phi = np.zeros(I)
# TODO: phi_j = suma sobre S sin j de |S|!(I-|S|-1)!/I! * (v(S u {j}) - v(S))

### 3b) (2 pts) Verifica eficiencia y compara con TreeSHAP

Verifica que $\sum_j \phi_j = f(\boldsymbol{x}^{*}) - \mathbb{E}[f]$, con la esperanza tomada sobre el fondo. Luego calcula los valores SHAP del mismo caso con `shap.TreeExplainer` en modo intervencional sobre el mismo fondo, y verifica que coinciden con los tuyos.

In [ ]:
# TODO: comprobar eficiencia

masker = shap.maskers.Independent(fondo, max_samples=len(fondo))
explicador = shap.TreeExplainer(modelo, data=masker, feature_perturbation="interventional",
                                model_output="raw")
# TODO: explicador.shap_values(caso) y comparacion con phi

### 3c) (2 pts) De lo local a lo global

Calcula los valores SHAP de todas las personas de validación con el explicador de 3b). Ordena los atributos por $\frac{1}{M}\sum_m |\phi_j^{(m)}|$ y compara ese orden con el de la importancia por permutación de 2a). Comenta las diferencias, si las hay.

In [ ]:
# TODO: shap_values de X_val, importancia media absoluta y comparacion de rankings

# Parte 4: Explicaciones locales y sus límites

## 4. LIME, contrafactuales y el modelo interpretable

### 4a) (3 pts) LIME, a mano

Para la misma persona de 3a), genera 1.000 perturbaciones sumando ruido gaussiano con la desviación estándar de cada atributo en entrenamiento. Recorta los conteos a valores no negativos y redondea `sex` y `c_charge_degree` a 0/1. Pondera cada perturbación con el kernel $\pi(\boldsymbol{z}) = \exp(-d^2/\sigma^2)$, con $d$ la distancia euclidiana en unidades estandarizadas y $\sigma = 1$. Ajusta un Lasso ponderado ($\alpha = 0{,}01$) de la predicción en log-odds sobre las desviaciones estandarizadas, y reporta sus coeficientes al lado de los $\phi_j$ de 3a).

In [ ]:
desv = X_tr.std().values
x0 = caso.values.ravel()

def lime(x0, sigma=1.0, n=1000, semilla=0):
    rng = np.random.default_rng(semilla)
    Z = x0 + rng.normal(size=(n, len(x0))) * desv
    # TODO: recortar conteos (columnas 2 a 5) a >= 0 y redondear/recortar sex y c_charge_degree a 0/1
    U = (Z - x0) / desv
    # TODO: pesos del kernel, prediccion f_raw sobre Z como DataFrame con columnas ATRIB
    # TODO: Lasso(alpha=0.01) ponderado sobre U; devolver los coeficientes
    pass

# TODO: tabla con los coeficientes de LIME y los phi de 3a)

### 4b) (2 pts) ¿Cuánto cambia la explicación?

Repite LIME con 20 semillas para cada $\sigma \in \{0{,}5;\ 1;\ 3\}$. Reporta la media y la desviación estándar de cada coeficiente en cada caso, y comenta cuál de los dos defectos de la diapositiva pesa más aquí: la variabilidad entre corridas o la elección del kernel.

In [ ]:
# TODO: tabla de medias y desviaciones por sigma

### 4c) (2 pts) Contrafactuales de un solo atributo

Toma las tres personas de validación con mayor probabilidad predicha. Para cada una y cada atributo por separado, busca el cambio más chico que lleva la probabilidad predicha bajo 0,5, recorriendo valores plausibles: edades hasta 80, conteos desde el valor actual hasta 0 y los binarios invertidos. Reporta una tabla con el cambio mínimo encontrado, o "ninguno" si no existe.

In [ ]:
top3 = X_val.iloc[np.argsort(-modelo.predict_proba(X_val)[:, 1])[:3]]

# TODO: para cada persona y atributo, recorrer los valores candidatos en orden de cercania
#       y guardar el primero que deja la probabilidad bajo 0.5

### 4d) (2 pts) ¿Sirven como recurso?

Con la tabla de 4c) y los tres requisitos de la diapositiva (ralo, accionable, plausible), discute si alguno de esos contrafactuales es algo que la persona podría hacer. ¿Qué dice eso sobre la idea de *recurso* cuando el modelo usa sobre todo atributos que no se pueden cambiar?

Respuesta 4d): completar aquí

### 4e) (2 pts) La posición de Rudin, con números

Ajusta una regresión logística con solo dos atributos, `age` y `priors_count`, y compara su AUC en validación con el del LightGBM de 1b) y el de COMPAS. Comenta qué implica el resultado para la decisión de usar un modelo de caja negra y explicarlo después.

In [ ]:
# TODO: logistica con age y priors_count, AUC en validacion contra modelo y compas

# Parte 5: ¿Trata igual a los dos grupos?

## 5. Equidad formal

### 5a) (3 pts) La matriz de confusión por grupo

Clasifica como positivo a quien COMPAS asigna `decile_score` $\ge 5$. Para tu modelo, usa el umbral que produce la **misma tasa global** de positivos en validación, para que la comparación sea justa. Para cada clasificador y cada grupo, reporta en validación la prevalencia, la tasa de positivos (paridad demográfica), el TPR, el FPR y el PPV.

In [ ]:
def tasas_por_grupo(pred, y, grupo):
    filas = []
    for g, nombre in [(1, "African-American"), (0, "Caucasian")]:
        m = grupo == g
        tn, fp, fn, tp = confusion_matrix(y[m], pred[m], labels=[0, 1]).ravel()
        # TODO: prevalencia, tasa de positivos, TPR, FPR y PPV
        pass
    return pd.DataFrame(filas)

g_val = grupo[i_val]
pred_compas = (compas[i_val] >= 5).astype(int)
# TODO: umbral del modelo con la misma tasa global de positivos que pred_compas
# TODO: tablas para COMPAS y para el modelo

### 5b) (2 pts) La imposibilidad, con números

Despejando el FPR de la expresión de calibración del apéndice se obtiene, para un grupo con prevalencia $p$:

$$\text{FPR} = \frac{p}{1-p}\cdot\frac{1-\text{PPV}}{\text{PPV}}\cdot\text{TPR}.$$

Usando la tabla de tu modelo en 5a), calcula qué FPR tendría que tener el grupo `Caucasian` para que su PPV **y** su TPR fueran iguales a los del grupo `African-American`, y compáralo con el FPR del grupo `African-American`.

In [ ]:
# TODO: FPR forzado para el grupo Caucasian con PPV y TPR del grupo African-American

### 5c) (2 pts) ProPublica contra la empresa

ProPublica comparó tasas de falsos positivos; la empresa respondió con el PPV por grupo. Con tus tablas de 5a) y el cálculo de 5b), di qué parte de cada argumento es correcta y por qué no podían ponerse de acuerdo sin que uno de los dos renunciara a su criterio.

Respuesta 5c): completar aquí

### 5d) (2 pts) Sacar la raza no la saca

Entrena un LightGBM, con los mismos hiperparámetros de 1b), que prediga el **grupo** a partir de los siete atributos del modelo. Reporta su AUC en validación y explica qué significa ese número para la estrategia de *fairness through unawareness* de 1b).

In [ ]:
# TODO: clasificador del grupo a partir de ATRIB y su AUC en validacion

### 5e) (3 pts) Un umbral por grupo

Siguiendo la idea de Hardt, elige un umbral distinto para cada grupo de modo que ambos tengan en validación el mismo FPR, igual al FPR global de tu modelo en 5a). Reporta para cada grupo el TPR y el PPV antes y después, y la accuracy global antes y después.

In [ ]:
# TODO: FPR global del modelo con el umbral de 5a)
# TODO: por grupo, umbral cuyo FPR en validacion queda mas cerca del objetivo (recorre roc_curve)
# TODO: tabla antes/despues

### 5f) (2 pts) ¿Qué se pagó, y quién decide?

Con la tabla de 5e): ¿qué criterio de equidad se ganó, cuál se perdió y cuánto costó en accuracy? Explica por qué elegir entre ellos es una decisión normativa y no técnica, y quién debería tomarla en un tribunal.

Respuesta 5f): completar aquí

# Parte 6: ¿Cuánto revelan los datos?

## 6. Privacidad

### 6a) (2 pts) Sacar los nombres no anonimiza

Sobre `df`, calcula qué fracción de las personas es **única** en el dataset para cada uno de estos conjuntos de cuasi-identificadores: (`sex`, `race`, `dob`), (`sex`, `race`, año de nacimiento) y (`sex`, `race`, `age_cat`). Reporta también el $k$ mínimo (el tamaño del grupo más chico) en cada caso. Solo se cuentan tamaños de grupos: no se busca a nadie.

In [ ]:
conjuntos = {
    "sexo, raza, fecha de nacimiento": ["sex", "race", "dob"],
    "sexo, raza, ano de nacimiento": ["sex", "race", "anio"],
    "sexo, raza, tramo de edad": ["sex", "race", "age_cat"],
}
d = df.assign(anio=df["dob"].str[:4])

# TODO: para cada conjunto, fraccion de filas cuyo grupo tiene tamano 1, y k minimo

### 6b) (3 pts) El mecanismo de Laplace

Se quiere publicar la tabla de conteos de reincidentes por grupo y `age_cat` (seis celdas). Agregar o quitar una persona cambia una sola celda en 1, así que la sensibilidad del vector completo es $\Delta = 1$. Simula 1.000 publicaciones con el mecanismo de Laplace para cada $\varepsilon \in \{0{,}1;\ 1;\ 10\}$, y grafica el error relativo medio de cada celda contra el tamaño de la celda, con una curva por $\varepsilon$.

In [ ]:
tabla = df[df["two_year_recid"] == 1].groupby(["race", "age_cat"]).size()
print(tabla)

rng = np.random.default_rng(0)
# TODO: para cada epsilon, 1000 publicaciones tabla + Laplace(0, 1/epsilon)
# TODO: error relativo medio por celda y grafico contra el conteo real (escala log en ambos ejes)

# Parte 7: Rendir cuentas

## 7. Lo que se le dice al tribunal

### 7a) (2 pts) La ficha del modelo

Escribe el párrafo que acompañaría a tu modelo si un tribunal pensara usarlo, al estilo de una *model card*: qué datos y qué etiqueta usa (y por qué esa etiqueta es un proxy), qué tan bien ordena comparado con COMPAS y con la logística de dos variables, qué criterio de equidad cumple y cuál no, qué tipo de explicación puede darle a la persona afectada, y la pregunta con que cierra el capítulo: a quién afecta cuando se equivoca y quién responde.

Respuesta 7a): completar aquí